# Step 7: many heads, and position

This notebook goes with step 7 of `vak`. It runs a small attention layer with many heads, then adds position.
`uv run vak heads` runs the same code at a bigger size, on real tokens.

Before this: step 6 runs one head on three tokens. After this: step 8 stacks these layers into the whole GPT.

| Section | Idea | Used in |
|---|---|---|
| 1 | Many heads: one attention per slice of each vector | `vak heads` |
| 2 | Position: without it, attention sees a set | `vak heads` |

## 1. Many heads

One head finds one kind of match. **Many heads** let the model look for several at once. One Linear makes the
queries, keys and values of every head together. Each vector is then cut into slices, one per **head**, and each
head runs the same `attention` on its own slice. The heads' outputs are joined side by side, and one more Linear
mixes them. Here a layer has vectors of 4 numbers and 2 heads, so each head reads 2 numbers.

In [1]:
import torch
from medha.model import SelfAttention, attention

torch.manual_seed(0)
layer = SelfAttention(width=4, heads=2)
x = torch.randn(1, 3, 4)  # one sentence of 3 tokens, 4 numbers each
q, k, v = layer.qkv(x)[0].split(4, dim=-1)
outputs = []
for head in range(2):
    part = slice(2 * head, 2 * head + 2)
    steps = attention(q[:, part], k[:, part], v[:, part])
    print(f"head {head} reads numbers {2 * head} and {2 * head + 1}; the last token's weights:",
          [round(number, 3) for number in steps.weights[-1].tolist()])
    outputs.append(steps.out)
by_hand = layer.out(torch.cat(outputs, dim=-1))
print("by hand, the last token:  ", [round(number, 3) for number in by_hand[-1].tolist()])
print("the layer, the last token:", [round(number, 3) for number in layer(x)[0, -1].tolist()])

head 0 reads numbers 0 and 1; the last token's weights: [0.339, 0.369, 0.292]
head 1 reads numbers 2 and 3; the last token's weights: [0.341, 0.26, 0.399]
by hand, the last token:   [-0.04, -0.391, -0.019, -0.777]
the layer, the last token: [-0.04, -0.391, -0.019, -0.777]


**What you see**

The two heads weigh the same three tokens differently: head 0 gives `[0.339, 0.369, 0.292]` and head 1 gives
`[0.341, 0.26, 0.399]`. Joined and mixed by the last Linear, they give `[-0.04, -0.391, -0.019, -0.777]` for the
last token, and the layer gives the same numbers in one pass: its view and transpose cut every head at once. The
work is split, not repeated: two heads of 2 numbers cost about what one head of 4 costs.

Used in: `uv run vak heads` runs a layer with vectors of 8 numbers and 2 heads, and logs the shape at each step.

## 2. Position

Attention mixes the tokens a token may read, by weight. The weights depend on the vectors, not on where the tokens
sit, so attention alone treats the earlier tokens as a set. **Position** fixes that. A second table holds one
learned row per place in the window, and each token's vector becomes its token row plus its position row. Here are
four toy tokens in two orders, "dog bites man ." and "man bites dog .", with the same last token.

In [2]:
from medha.model import Embedding

words = ["dog", "bites", "man", "."]  # a token's id is its place in this list
torch.manual_seed(0)
embedding = Embedding(vocab_size=4, context=4, width=4)
layer = SelfAttention(width=4, heads=2)
dog_bites_man = torch.tensor([[0, 1, 2, 3]])
man_bites_dog = torch.tensor([[2, 1, 0, 3]])
for label, vectors in [("without position", embedding.tokens), ("with position", embedding)]:
    first = layer(vectors(dog_bites_man))[0, -1]
    second = layer(vectors(man_bites_dog))[0, -1]
    print(f"{label:17} dog bites man . -> {[round(number, 3) for number in first.tolist()]}")
    print(f"{label:17} man bites dog . -> {[round(number, 3) for number in second.tolist()]}")

without position  dog bites man . -> [-0.045, 0.027, -0.088, 0.06]
without position  man bites dog . -> [-0.045, 0.027, -0.088, 0.06]
with position     dog bites man . -> [0.187, 0.18, 0.16, -0.062]
with position     man bites dog . -> [0.038, 0.065, 0.08, -0.142]


**What you see**

Without position, the last token "." gets `[-0.045, 0.027, -0.088, 0.06]` in both orders: it reads the same four
vectors, so it gets the same mix. With position, the two orders give different outputs:
`[0.187, 0.18, 0.16, -0.062]` for "dog bites man ." and `[0.038, 0.065, 0.08, -0.142]` for "man bites dog .". Now
the model can tell who bit whom.

Used in: `uv run vak heads` shows the same with real tokens, "The dog saw the cat." and "The cat saw the dog.".
The GPT in step 8 starts with this `Embedding`.

## Recap

| Idea | Remember | Used next |
|---|---|---|
| Many heads | one Linear makes q, k and v; each head runs attention on its slice; one more Linear mixes the heads | `SelfAttention` in every block of the GPT, step 8 |
| The shapes | (batch, tokens, width) becomes (batch, heads, tokens, width / heads) by view and transpose | the KV cache, step 10, stores keys and values in this shape |
| Position | a learned row per place, added to the token row; without it, "dog bites man ." and "man bites dog ." end the same | the GPT's first part, `Embedding`, step 8 |